# 01 — Routing and Jev

Adaptive routing (council condition F) and the Jev bounded-decision provider. Two different benchmarks: routing was measured on the 71 council evaluation cases; Jev on the decision corpus `decision-corpus-v1`. The two are not compared with each other. Labels used in every notebook: **MEASURED** (a real provider or real run produced the number), **ESTIMATED** (derived here from measured data by a stated formula), **SIMULATED** (mock runner, no real model), **NOT_RUN**, **UNKNOWN** (the evidence does not say). Implementation and wiring are reported separately from evaluation: code that exists is not a measured capability.

In [1]:
import json
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 11  # the pre-registered bootstrap seed; reused for every derived computation here
random.seed(SEED)
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EVIDENCE_MAP.md").exists())  # research/council-ablation
EV = ROOT / "evidence"
JEV = ROOT.parent / "jev" / "evidence"  # the JEV study's canonical copy in this repository
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def load(rel):
    return json.loads((EV / rel).read_text())


def jsonl(rel):
    return [json.loads(x) for x in (EV / rel).read_text().splitlines() if x.strip()]


def final_calls(rel):
    """The harness cache is append-only: a re-asked call (Amendments 1 and 2) appends a new row and the
    last row per (member, case) is the one analysed. Earlier rows stay in the file as recorded calls."""
    return pd.DataFrame(jsonl(rel)).drop_duplicates(["member", "case_id"], keep="last").reset_index(drop=True)


R = load("council/results.json")
REG = {r["id"]: r for r in load("reasoning_methods/reasoning_methods_registry.json")}
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("evidence files:", sorted(str(p.relative_to(ROOT)) for p in EV.rglob("*") if p.is_file()))


python 3.12.12 | pandas 3.1.0 | numpy 2.5.4 | seed 11
evidence files: ['evidence/SOURCE_MANIFEST.json', 'evidence/council/calibration_by_member.csv', 'evidence/council/cases.csv', 'evidence/council/cases.json', 'evidence/council/conditions.csv', 'evidence/council/cost_latency_by_condition.csv', 'evidence/council/judge_calls.csv', 'evidence/council/judge_calls.jsonl', 'evidence/council/member_calls.csv', 'evidence/council/member_calls.jsonl', 'evidence/council/member_calls_budget300_archive.csv', 'evidence/council/member_calls_budget300_archive.jsonl', 'evidence/council/results.json', 'evidence/decision_methods/ablation-2026-10-20.json', 'evidence/notebook_runs.json', 'evidence/publication_scan.json', 'evidence/reasoning_methods/method_status_summary.csv', 'evidence/reasoning_methods/reasoning_methods_registry.json', 'evidence/reasoning_methods/reasoning_strategies_mock_benchmark.md', 'evidence/reasoning_methods/research_plans.json', 'evidence/replay_check.json']


In [2]:
#: Evaluation state per method, with the evidence that sets it. Explicit, not inferred from text.
EVALUATION = {
    "deterministic-baseline": "MEASURED",      # reference arm of the decision-methods ablation
    "single-strong-model": "MEASURED",         # condition A of the council benchmark
    "council": "MEASURED",                     # conditions B1-D, F (harness, not the product endpoint)
    "proposer-judge": "MEASURED",              # as condition E (judge over C) in the harness only
    "critic-veto": "NOT_RUN",
    "critic-selective-council": "NOT_RUN",
    "cot": "SIMULATED (mock runner)",
    "tot": "SIMULATED (mock runner)",
    "got": "SIMULATED (mock runner)",
    "self-consistency": "NOT_RUN",
    "react": "NOT_RUN",
    "reflexion": "NOT_RUN",
    "pot-evidence": "NOT_RUN",
    "mcts": "NOT_RUN",
    "jev": "MEASURED",                         # real provider calls on decision-corpus-v1
    "agentic-llm-decision": "MEASURED",        # decision-methods ablation, small local model
    "hermes-agent-patterns": "NOT_RUN",
}


def status_row(method_id):
    """Implementation / wiring / evaluation labels, derived from the registry snapshot only."""
    r = REG[method_id]
    implemented = "NOT_IMPLEMENTED" if r["wiring"] == "NOT_PRESENT" else "IMPLEMENTED"
    wired = "WIRED" if r["wiring"] in {"WIRED", "DECISION_AFFECTING", "OBSERVED", "EVALUATED"} else "NOT_WIRED"
    evaluation = EVALUATION[method_id]
    return {"method": r["name"], "implemented": implemented, "wired": wired, "authority": r["authority"],
            "evaluation": evaluation, "production": r["production"], "recommendation": r["recommendation"],
            "result": r["result"], "limitations": r["limitations"]}


In [3]:
pd.DataFrame([status_row(m) for m in ("single-strong-model", "jev", "agentic-llm-decision", "hermes-agent-patterns")]).T

,0,1,2,3
method,Single strong reasoning model (reasoning-depth router),"Jev bounded decisions (typed, closed-set)",Agentic LLM decision method,Hermes Agent patterns (Nous Research)
implemented,IMPLEMENTED,IMPLEMENTED,IMPLEMENTED,NOT_IMPLEMENTED
wired,WIRED,WIRED,WIRED,NOT_WIRED
authority,ADVISORY,ADVISORY,ADVISORY,NONE
evaluation,MEASURED,MEASURED,MEASURED,NOT_RUN
production,IN_PRODUCT,IN_PRODUCT_OFF_BY_DEFAULT,IN_PRODUCT_OFF_BY_DEFAULT,NOT_IN_PRODUCT
recommendation,ADOPT,RESEARCH,SELECTIVE,RESEARCH
result,"Frontier single models 0.958–0.986; local Qwen 7B 0.620, 1.5B 0.282 — model strength dominates council effects.","EXECUTION_SCALE (n=90): accuracy 0.578 → 0.733, Brier 0.576 → 0.364 with typed semantics; 95% Wilson intervals overlap.",In-process bounded methods are faster and model-independent for bounded decisions; agentic arm was a small local LLM.,UNKNOWN
limitations,Ceiling effect on the 71-case set.,"One decision family, one model; vendor dependency; shadow-mode comparison against the router not built.",Run SHA missing in the evidence file; small agentic model.,Never a second agent runtime inside Athena.


## Adaptive routing (condition F) — MEASURED

Fixed rule, not tuned: agreement of two cheap members → answer; disagreement → judge; judge confidence < 0.6 or a high-consequence case → council D and human review.

In [4]:
F = R["conditions"]["F_adaptive_routing"]
pd.Series({k: F[k] for k in ("accuracy", "critical_miss_rate", "calls_per_decision", "tokens_per_decision",
                              "latency_s_per_decision", "cost_usd_per_decision")} | {"delta_vs_strong_solo": F["vs_strong_solo"]["delta"],
                              "ci95": F["vs_strong_solo"]["ci95"], "decision": R["decisions"]["F_adaptive_routing"]}, name="F").to_frame()

,F
accuracy,0.9718
critical_miss_rate,0.018519
calls_per_decision,3.35
tokens_per_decision,1500.6
latency_s_per_decision,3.07
cost_usd_per_decision,0.002029
delta_vs_strong_solo,-0.0141
ci95,"[-0.0423, 0.0]"
decision,REJECTED


In [5]:
pd.Series(R["hypotheses"]["H3"], name="H3").to_frame()

,H3
F_accuracy,0.9718
F_tokens,1500.6
F_latency,3.07
best_always_on,C_multi_provider
best_always_accuracy,0.9859
best_always_tokens,1187.4
best_always_latency,2.71
F_vs_best_always,"{'delta': -0.0141, 'ci95': ['-0.0423', '0.0']}"
F_decision,REJECTED


**RESULT.** F scored 0.972 (69/71) against 0.986 for the strong solo model and used more tokens than the multi-provider council, because every high-consequence case was routed to council D. H3 is rejected on this benchmark. The routing rule of the Athena *Hermes* persona is not part of this measurement: routing quality for that persona is `NOT_RUN`. The registry row `Hermes Agent patterns (Nous Research)` above is a different thing: an external agent runtime whose patterns were reviewed; it is not embedded in Athena and is not the Athena Hermes persona.

## Jev bounded decisions — MEASURED (real provider calls)

Jev was asked the same typed, closed-set question as every other decision method; Athena's policy resolver had the final say. Provider cost is `UNKNOWN`: the API reports tokens but no price.

In [6]:
A = json.loads((JEV / "jev-ablation-2026-10-21.json").read_text())
rows = []
for dtype, res in A["results"].items():
    for arm, m in res["arms"].items():
        rows.append({"decision_type": dtype, "arm": arm, "n": m.get("cases_evaluated"), "accuracy_raw": m.get("accuracy_raw"),
                     "accuracy_final": m.get("accuracy_after_resolver_with_fallback"), "brier": m.get("brier"), "ece": m.get("ece")})
jev = pd.DataFrame(rows)
jev.pivot_table(index="decision_type", columns="arm", values="accuracy_raw").round(3)

arm,agentic,deterministic,hybrid,jev,local_bounded
decision_type,,,,,
EXECUTION_SCALE,0.678,0.911,0.911,0.578,0.922
EXECUTION_STRATEGY,0.086,1.000,0.829,0.400,0.886
EXPLANATION_MODE,0.484,0.613,0.742,0.226,0.774
HPO_DEPTH,0.323,0.064,0.548,0.194,0.516
HUMAN_REVIEW,0.790,1.000,0.895,0.825,0.895
RETRY_FALLBACK,0.314,0.943,0.857,0.400,0.857


In [7]:
print("calls:", A["jev_calls"], "| cost:", A["cost"]["state"], "-", A["cost"]["reason"])
for line in A["limitations"]:
    print("-", line)

calls: 309 | cost: UNKNOWN - TypeSafe returns token usage but no price or balance; Athena does not invent one. Output tokens are documented as free of charge.
- MEASURED labels (HPO_DEPTH, EXPLANATION_MODE) come from synthetic datasets on one Apple-silicon machine; they are hardware-specific.
- POLICY labels are the governance rule's own answer: rules are correct by definition there; Jev is judged on how well it approximates policy, not on beating it.
- Jev received the decision description, the typed features and the candidate action names — the same information the agentic method gets; no Jev-specific hints were written.
- Agentic and hybrid numbers are the recorded 2026-10-20 baseline on the same split, not re-run.


## Jev with Athena semantic context — MEASURED, one decision family

In [8]:
S = json.loads((JEV / "jev-semantics-ablation-2026-10-21.json").read_text())
cmp = S["comparison_vs_experiment_a"]["EXECUTION_SCALE"]
pd.DataFrame({k: v for k, v in cmp.items() if isinstance(v, dict) and "A" in v}).T[["A", "B", "delta_B_minus_A"]]

,A,B,delta_B_minus_A
abstention_rate,0.0,0.0,0.0
accuracy_after_resolver_with_fallback,0.6,0.7333,0.1333
accuracy_raw,0.5778,0.7333,0.1555
brier,0.5756,0.3636,-0.212
ece,0.1962,0.1828,-0.0134
failure_rate,0.0,0.0,0.0
input_tokens_per_call,453.6,807.6,NaN
latency_ms_p50,253.038,121.337,-131.701
latency_ms_p95,316.463,227.681,-88.782
mean_confidence,0.3113,0.5196,0.2083


In [9]:
pd.Series(S["token_accounting"]["A"]).to_frame("A").join(pd.Series(S["token_accounting"]["B"]).to_frame("B"))

,A,B
evaluated_calls,279.0,279.0
input_tokens,114560.0,183524.0
input_tokens_per_call,410.6,657.8
output_tokens,16947.0,16731.0


**RESULT.** On `EXECUTION_SCALE` (n = 90), typed semantics raised Jev's raw accuracy from 0.578 to 0.733 and lowered Brier from 0.576 to 0.364, at 1.78× input tokens per call on that family (1.60× over all 279 evaluated calls of the six families). The 95 % Wilson intervals overlap, so the gain is not established. Deterministic rules remain more accurate on policy-labelled decisions by construction (the policy *is* the label).

**Not measured:** Jev latency on a comparable network path, Jev on the council case set, shadow-mode comparison against the router.